Messy code for counting tokens from the experiments.

In [1]:

from collections import Counter
from pathlib import Path

from redel.utils import read_jsonl

In [2]:
# define base experiments path
EXPERIMENTS = Path("./pal-logs/instances")
ANALYSIS_PROMPT = (
    "Analyze a transcript from a doctor-patient encounter and provide actionable communication improvement advice for"
    " the doctor."
)

In [3]:
# from the docs
def count(fp):
    tokens_used_prompt = Counter()
    tokens_used_output = Counter()
    msg_counts = Counter()
    characters_11labs = 0
    counting_11labs = "VOICE" in str(fp)

    for event in read_jsonl(fp):
        # when we see the feedback prompt, stop counting 11labs
        if (
            event["type"] == "kani_message"
            and event["msg"]["role"] == "user"
            and event["msg"]["content"].startswith(ANALYSIS_PROMPT)
        ):
            counting_11labs = False
        # count user messages
        if event["type"] == "kani_message" and not event["msg"]["content"].startswith(ANALYSIS_PROMPT):
            msg_counts[event["msg"]["role"]] += 1
        # count characters
        if event["type"] == "kani_message" and event["msg"]["role"] == "assistant" and counting_11labs:
            characters_11labs += len(event["msg"]["content"])
        # count tokens
        if event["type"] == "tokens_used":
            tokens_used_prompt[event["id"]] += event["prompt_tokens"]
            tokens_used_output[event["id"]] += event["completion_tokens"]

    return tokens_used_prompt, tokens_used_output, characters_11labs, msg_counts

In [9]:
for user_instance in EXPERIMENTS.iterdir():
    if not user_instance.is_dir():
        # print(f"not a dir: {user_instance}")
        continue
    for session in user_instance.iterdir():
        if not session.is_dir():
            # print(f"not a dir: {session}")
            continue

        tokens_used_prompt, tokens_used_output, characters_11labs, msg_counts = count(session / "events.jsonl")
        if not msg_counts:
            continue
        print(f"{user_instance.name},{session.name},{tokens_used_prompt.total()},{tokens_used_output.total()},{characters_11labs},{msg_counts['user']}")

doqo94daq8,1736867710-aiden-941dba03-3fe9-4b99-b761-77f84703108d,13519,801,0,7
doqo94daq8,1736867710-nicki_VOICE-ffa31424-7de1-4bfd-9373-a1cdbd0511aa,26019,913,1415,13
lqpyx3xf44g,1733012437-nicki_VOICE-9a36e026-5a7f-44c7-94d0-10552256e7ab,26830,764,678,12
lqpyx3xf44g,1733012437-aiden-42eba8eb-6166-4a3e-86cf-b58674a4c924,13646,739,0,7
lqpyx3xf44g,1733012437-aaron-d5ba80c3-a5a4-4399-9d29-37d2440f9db9,2291,82,0,2
yzkge306jj,1736364935-aiden_VOICE-2898717a-7587-445a-84b9-de9638cc8f27,3899,53,235,3
jahcgq0tee,1733878932-aaron-ca104c4f-2be9-4bf9-b8b3-e250be68d497,1103,41,0,1
jahcgq0tee,1733878932-aiden_VOICE-e39fd0f0-2251-4237-8b89-99c06ccc6b76,1255,26,112,1
jahcgq0tee,1733878932-nicki_VOICE-09b8d1d8-e6fe-4d15-ba60-4384e0db7d23,1374,17,73,1
jahcgq0tee,1733878932-aiden-229bb112-25e5-428f-8f12-f67367bfb523,17439,463,0,11
ypzb9vb6w2,1732660398-aaron-a2c1d78b-3022-4e93-9e3b-c5e0cba45616,1104,59,0,1
ypzb9vb6w2,1732659493-aaron-21296f62-1d46-4d89-a45a-b3050e198653,2258,87,0,2
uz6va1mo75,173489242